# Notebook 03: our GA from Lab Task 2
This is our GA from Task 2 without changes. Every 0.25 s pygad evolves a target (x, y) for 8
generations with population 20, using our Task 2 fitness (3 nearest coins, travel cost, monster and fire
penalties along the path, stagnation boost). It runs on the same worlds as the other two agents.

In [1]:
# mount google drive and make the folders we save everything into
import os, sys, json, subprocess, datetime, importlib
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/COMP6009_Assignment-Part A'   # all our files go here
PATHS = {
    'code':         f'{ROOT}/code',                          # the cointex_parta package
    'data':         f'{ROOT}/data',                          # cointex.kv, level table
    'config':       f'{ROOT}/config',                        # experiment_config.json
    'greedy':       f'{ROOT}/results/01_greedy_baseline',
    'proposed':     f'{ROOT}/results/02_proposed',
    'ga':           f'{ROOT}/results/03_ga_task2',
    'compare':      f'{ROOT}/results/04_comparison',
    'report_figs':  f'{ROOT}/report_assets/figures',         # final figures for the report/slides
    'report_tabs':  f'{ROOT}/report_assets/tables',          # final tables for the report
    'deliverables': f'{ROOT}/deliverables',
    'logs':         f'{ROOT}/logs',
}
for p in PATHS.values():
    os.makedirs(p, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pygad==3.3.1'], check=True)
if PATHS['code'] not in sys.path:
    sys.path.insert(0, PATHS['code'])

def log_run(notebook, **info):
    # keep a log of every run
    info.update(notebook=notebook, at=datetime.datetime.now().isoformat(timespec='seconds'))
    with open(f"{PATHS['logs']}/run_log.jsonl", 'a') as f:
        f.write(json.dumps(info) + '\n')

print('Project root:', ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/COMP6009_Assignment-Part A


In [2]:
# load our code (written to drive by notebook 00) and the experiment settings
assert os.path.exists(f"{PATHS['code']}/cointex_parta/__init__.py"), \
    'code not found on Drive (notebook 00 writes it)'
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
import cointex_parta
from cointex_parta import runner, plots, sim, levels
plots.style()
CFG = json.load(open(f"{PATHS['config']}/experiment_config.json"))
LEVELS, SEEDS = CFG['levels'], list(range(CFG['n_seeds']))
WORKERS = os.cpu_count()
print(f"cointex_parta v{cointex_parta.__version__} | {len(LEVELS)} levels x {len(SEEDS)} seeds | {WORKERS} CPU workers")
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

for sub in ('figures', 'tables'): os.makedirs(f"{PATHS['ga']}/{sub}", exist_ok=True)
OUT = PATHS['ga']

cointex_parta v1.0.0 | 24 levels x 50 seeds | 12 CPU workers


## 1. Running the GA on all 24 levels
The GA is the slowest of the three because it scores a lot of candidate positions every move.

In [3]:
CSV = f'{OUT}/per_run.csv'
runner.run_sweep(CSV, LEVELS, SEEDS, ['ga_task2'], workers=WORKERS)
df = pd.read_csv(CSV)
log_run('03_ga', episodes=len(df))
overall = runner.summarise(df); per_level = runner.summarise(df, by=('agent', 'level'))
overall.to_csv(f'{OUT}/tables/overall_summary.csv', index=False)
per_level.to_csv(f'{OUT}/tables/per_level_summary.csv', index=False)
display(overall.round(2))
display(per_level[['level', 'success_rate', 'death_rate', 'completion_time_mean', 'remaining_life_mean',
                   'near_miss_pct_mean', 'plan_ms_mean']].round(2))

0 episodes already in per_run.csv; running 1200 more
  60/1200 episodes  (47.05/s, ETA 0.4 min)
  120/1200 episodes  (46.10/s, ETA 0.4 min)
  180/1200 episodes  (43.56/s, ETA 0.4 min)
  240/1200 episodes  (38.51/s, ETA 0.4 min)
  300/1200 episodes  (35.53/s, ETA 0.4 min)
  360/1200 episodes  (33.79/s, ETA 0.4 min)
  420/1200 episodes  (33.23/s, ETA 0.4 min)
  480/1200 episodes  (32.96/s, ETA 0.4 min)
  540/1200 episodes  (32.03/s, ETA 0.3 min)
  600/1200 episodes  (31.56/s, ETA 0.3 min)
  660/1200 episodes  (31.31/s, ETA 0.3 min)
  720/1200 episodes  (30.45/s, ETA 0.3 min)
  780/1200 episodes  (29.42/s, ETA 0.2 min)
  840/1200 episodes  (29.42/s, ETA 0.2 min)
  900/1200 episodes  (27.31/s, ETA 0.2 min)
  960/1200 episodes  (26.33/s, ETA 0.2 min)
  1020/1200 episodes  (25.47/s, ETA 0.1 min)
  1080/1200 episodes  (24.71/s, ETA 0.1 min)
  1140/1200 episodes  (24.39/s, ETA 0.0 min)
  1200/1200 episodes  (23.71/s, ETA 0.0 min)


,agent,episodes,success_rate,death_rate,timeout_rate,avg_coin_pct,completion_time_mean,completion_time_median,episode_time_mean,remaining_life_mean,near_miss_pct_mean,racr_mean,path_length_mean,plan_ms_mean
0,ga_task2,1200,100.0,0.0,0.0,100.0,6.99,6.22,6.99,93.28,1.62,2.06,4.98,16.26


,level,success_rate,death_rate,completion_time_mean,remaining_life_mean,near_miss_pct_mean,plan_ms_mean
0,1,100.0,0.0,3.42,99.60,0.31,14.29
1,2,100.0,0.0,3.95,99.33,0.39,14.10
2,3,100.0,0.0,4.90,97.67,1.08,14.51
3,4,100.0,0.0,4.78,97.80,0.92,15.31
4,5,100.0,0.0,5.97,94.80,1.00,17.85
5,6,100.0,0.0,5.49,96.10,0.84,19.00
6,7,100.0,0.0,6.64,92.00,1.30,15.91
7,8,100.0,0.0,6.19,95.28,1.19,14.77
8,9,100.0,0.0,5.82,94.72,0.85,14.57
9,10,100.0,0.0,6.50,91.67,2.56,15.23


In [4]:
for metric, ylab, title, name, kw in [
        ('won', 'Success rate (%)', 'GA (Task 2): success rate by level', 'ga_success_by_level.png', dict(scale=100, ylim=(0, 105))),
        ('completion_time', 'Seconds', 'GA (Task 2): completion time of won episodes', 'ga_time_by_level.png', {}),
        ('remaining_life', 'Remaining life (%)', 'GA (Task 2): remaining life', 'ga_life_by_level.png', dict(ylim=(0, 105)))]:
    fig = plots.per_level_bars(df, metric, ['ga_task2'], ylab, title, **kw)
    plots.save(fig, f'{OUT}/figures', name); plt.show()
row, trace = runner.play_episode(23, 0, 'ga_task2', record=True)
gif = plots.animate_episode(trace, 'ga_task2', f'{OUT}/figures/ga_L23_s0.gif')
display(Image(filename=gif))

Output hidden; open in https://colab.research.google.com to view.

In [5]:
headline = {
    'agent': 'ga_task2', 'episodes': int(len(df)),
    'success_rate_pct': round(100 * df.won.mean(), 2),
    'levels_always_won': int((df.groupby('level').won.mean() == 1).sum()),
    'completion_time_mean_s': round(df.completion_time.mean(), 2),
    'remaining_life_mean_pct': round(df.remaining_life.mean(), 2),
    'plan_ms_mean': round(df.mean_plan_ms.mean(), 2),
}
json.dump(headline, open(f'{OUT}/headline.json', 'w'), indent=2)
headline

{'agent': 'ga_task2',
 'episodes': 1200,
 'success_rate_pct': np.float64(100.0),
 'levels_always_won': 24,
 'completion_time_mean_s': np.float64(6.99),
 'remaining_life_mean_pct': np.float64(93.28),
 'plan_ms_mean': np.float64(16.26)}